# Pengujian Dataset HANDS — Open-Set Evaluation of a Hand-Gesture Robot Control Interface

Notebook ini menguji `HANDS` dataset (Mendeley Data, DOI `10.17632/ndrczc35bt.2`) sesuai kebutuhan `progres_robotika.pdf` (Kelompok 3, Teknik Informatika Unhas).

Alur:
1. **Verifikasi struktur & integritas** dataset (frame, hand-crop, label, defect path).
2. **Ekstraksi landmark** (MediaPipe HandLandmarker, Layer L2) dengan normalisasi wrist→origin.
3. **Latih MLP** (63→64→32→15) pada landmark.
4. **Bandingkan split LOSO vs random** — reproduksi H5 paper (96,22% vs 99,68%).

## 0. Konfigurasi

In [1]:
import os, sys, csv, json, re, time
from collections import Counter, defaultdict

import numpy as np
import cv2

# --- path dataset ---
HANDS = r"C:\Users\kasmi\OneDrive\Dokumen\GitHub\robotik_jurnal_rilis\dataset_uji\HANDS"
ANN = os.path.join(HANDS, "annotations")
EXTRACT = os.path.join(HANDS, "extracted")
LANDMARKS = os.path.join(HANDS, "landmarks")
MODEL_PATH = os.path.join(HANDS, "scripts", "models", "hand_landmarker.task")

print("HANDS path exists:", os.path.isdir(HANDS))
print("Model hand_landmarker.task exists:", os.path.exists(MODEL_PATH))

HANDS path exists: True
Model hand_landmarker.task exists: True


## 1. Verifikasi dataset

Cek jumlah frame, hand-crop, label unik, dan defect path anotasi.

In [2]:
def parse_ann(path):
    """Parse file anotasi CSV HANDS -> (labels, rows)."""
    labels, rows = [], []
    with open(path, encoding="utf-8", errors="ignore") as f:
        for i, row in enumerate(csv.reader(f)):
            if not row:
                continue
            if i == 0:
                labels = [x.strip() for x in row]
                continue
            if len(row) < 2:
                continue
            boxes = {}
            for j, lab in enumerate(labels):
                if j + 2 >= len(row):
                    continue
                cell = row[j + 2].strip()
                m = re.findall(r"-?\d+(?:\.\d+)?", cell)
                if len(m) == 4:
                    x, y, w, h = map(float, m)
                    if w > 0 and h > 0:
                        boxes[lab] = (x, y, w, h)
            rows.append({"rgb": row[0].strip(), "boxes": boxes})
    return labels, rows

summary = {}
for sub in range(1, 6):
    labels, rows = parse_ann(os.path.join(ANN, f"Subject{sub}.txt"))
    n_crops = sum(len(r["boxes"]) for r in rows)
    # defect path
    defect = None
    if rows:
        m = re.search(r"[Ss]ubject\s*(\d)", rows[0]["rgb"])
        if m and int(m.group(1)) != sub:
            defect = f"path->Subject{m.group(1)}"
    summary[sub] = {"rows": len(rows), "crops": n_crops, "unique_labels": len(set(labels)), "defect": defect}

total_rows = sum(v["rows"] for v in summary.values())
total_crops = sum(v["crops"] for v in summary.values())

print(f"{'Subjek':<8}{'Baris':<8}{'Hand-crop':<12}{'Label unik':<12}Defect")
for sub in range(1, 6):
    v = summary[sub]
    print(f"{sub:<8}{v['rows']:<8}{v['crops']:<12}{v['unique_labels']:<12}{v['defect']}")
print("\nTOTAL baris :", total_rows, "(paper: 12.000)")
print("TOTAL hand-crop:", total_crops, "(paper: 21.750)")

Subjek  Baris   Hand-crop   Label unik  Defect
1       2400    4350        31          path->Subject2
2       2400    4350        31          path->Subject3
3       2400    4350        31          path->Subject4
4       2400    4350        31          path->Subject5
5       2400    4350        31          path->Subject6

TOTAL baris : 12000 (paper: 12.000)
TOTAL hand-crop: 21750 (paper: 21.750)


### Hasil verifikasi
- Baris total = 12.000 → sesuai klaim (2400/subjek × 5).
- Hand-crop total = 21.750 → sesuai klaim.
- **Defect path terkonfirmasi**: tiap `SubjectN.txt` menunjuk ke `Subject{N+1}` → harus mengabaikan folder pada path, hanya pakai nama file.

## 2. Ekstraksi landmark (MediaPipe HandLandmarker)

Layer L2 paper: MediaPipe → 21 keypoint × 3 koordinat, diterjemahkan ke wrist & diskala dengan jarak wrist-ke-keypoint terbesar.

In [3]:
import mediapipe as mp
from mediapipe.tasks.python import vision

base_options = mp.tasks.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
landmarker = vision.HandLandmarker.create_from_options(options)

In [4]:
def resolve_rgb_path(p):
    # abaikan folder pada path (defect), cari berdasarkan nama file
    m = re.search(r"/(\d+)_color\.png$", p.replace("\\", "/"))
    if not m:
        return None
    fname = m.group(1)
    for sub in range(1, 6):
        cand = os.path.join(EXTRACT, f"Subject{sub}_RGB", f"Subject{sub}", f"{fname}_color.png")
        if os.path.exists(cand):
            return cand
    return None

def normalize_landmarks(lm_xyz):
    wrist = lm_xyz[0]
    d = lm_xyz - wrist
    scale = np.linalg.norm(d, axis=1).max()
    return d / max(scale, 1e-6)

def extract_landmarks(subject, limit=0):
    labels, rows = parse_ann(os.path.join(ANN, f"Subject{subject}.txt"))
    if limit:
        rows = rows[:limit]
    os.makedirs(os.path.join(LANDMARKS, f"Subject{subject}"), exist_ok=True)
    records = []  # (crop_idx, label, landmarks (21,3))
    crop_idx = 0
    detected = 0
    for r in rows:
        img_path = resolve_rgb_path(r["rgb"])
        if not img_path:
            continue
        img = cv2.imread(img_path)
        if img is None:
            continue
        for lab, (x, y, w, h) in r["boxes"].items():
            x0, y0 = max(0, int(x)), max(0, int(y))
            x1, y1 = min(img.shape[1], int(x + w)), min(img.shape[0], int(y + h))
            if x1 <= x0 or y1 <= y0:
                continue
            crop = img[y0:y1, x0:x1]
            rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
            if res.hand_landmarks:
                lm = res.hand_landmarks[0]
                pts = np.array([[lm_.x, lm_.y, lm_.z] for lm_ in lm])
                pts[:, 0] = pts[:, 0] * (x1 - x0) + x0
                pts[:, 1] = pts[:, 1] * (y1 - y0) + y0
                norm = normalize_landmarks(pts)
                np.save(os.path.join(LANDMARKS, f"Subject{subject}", f"crop_{crop_idx}.npy"), norm)
                records.append((crop_idx, lab, norm))
                detected += 1
            crop_idx += 1
    return records, detected, crop_idx

# UJI CEPAT: hanya Subject1, 30 baris pertama
records, detected, total = extract_landmarks(1, limit=30)
print(f"Subject1 (30 baris): {total} crop, {detected} terdeteksi ({detected/max(total,1)*100:.2f}%)")

Subject1 (30 baris): 53 crop, 52 terdeteksi (98.11%)


### Catatan ekstraksi
- Dengan `--full` (tanpa `limit`), pipeline memproses seluruh 21.750 crop. Proses ini butuh waktu (± menit–jam tergantung CPU).
- File landmark normalisasi tersimpan di `HANDS/landmarks/SubjectN/crop_i.npy`.

## 3. Latih MLP (63→64→32→15)

Menggunakan landmark normalisasi sebagai fitur. Di sini dipakai MLP sederhana (tanpa framework besar) agar mudah diverifikasi. Gunakan landmark dari ekstraksi penuh (bukan cuma 30 baris).

In [5]:
def load_all_landmarks(limit_per_subject=0):
    """Muat semua landmark + label dari folder landmarks."""
    X, y, subj = [], [], []
    for sub in range(1, 6):
        # dapatkan label per crop dari anotasi (dengan mengabaikan defect path)
        labels, rows = parse_ann(os.path.join(ANN, f"Subject{sub}.txt"))
        crop_labels = {}
        idx = 0
        for r in rows:
            for lab in r["boxes"]:
                crop_labels[idx] = lab
                idx += 1
        folder = os.path.join(LANDMARKS, f"Subject{sub}")
        files = sorted([f for f in os.listdir(folder) if f.endswith(".npy")])
        if limit_per_subject:
            files = files[:limit_per_subject]
        for fn in files:
            ci = int(re.search(r"crop_(\d+)\.npy", fn).group(1))
            X.append(np.load(os.path.join(folder, fn)).reshape(-1))
            y.append(crop_labels[ci])
            subj.append(sub)
    return np.array(X), np.array(y), np.array(subj)

# Catatan: jalankan setel ekstraksi penuh selesai.
# X, y, subj = load_all_landmarks()
# print("Dataset landmark:", X.shape, "- kelas unik:", len(set(y)))

In [6]:
# MLP ringan (numpy saja) untuk verifikasi cepat
class MLP:
    def __init__(self, sizes, lr=1e-2, epochs=40):
        self.sizes = sizes
        self.lr = lr
        self.epochs = epochs
        self.W, self.b = [], []
        rng = np.random.default_rng(42)
        for i in range(len(sizes) - 1):
            self.W.append(rng.standard_normal((sizes[i], sizes[i+1])) * np.sqrt(2/sizes[i]))
            self.b.append(np.zeros(sizes[i+1]))

    def forward(self, X):
        a = X
        for i in range(len(self.W) - 1):
            a = np.maximum(0, a @ self.W[i] + self.b[i])  # ReLU
        z = a @ self.W[-1] + self.b[-1]
        e = np.exp(z - z.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

    def train(self, X, y):
        Y = np.eye(self.sizes[-1])[y]
        for ep in range(self.epochs):
            a = X
            acts = [X]
            for i in range(len(self.W) - 1):
                a = np.maximum(0, a @ self.W[i] + self.b[i])
                acts.append(a)
            z = a @ self.W[-1] + self.b[-1]
            e = np.exp(z - z.max(axis=1, keepdims=True))
            p = e / e.sum(axis=1, keepdims=True)
            dz = p - Y
            dW = [acts[-1].T @ dz / len(X)]
            db = [dz.mean(axis=0)]
            for i in range(len(self.W) - 2, -1, -1):
                da = dz @ self.W[i+1].T
                da[acts[i+1] <= 0] = 0
                dz = da
                dW.insert(0, acts[i].T @ dz / len(X))
                db.insert(0, dz.mean(axis=0))
            for i in range(len(self.W)):
                self.W[i] -= self.lr * dW[i]
                self.b[i] -= self.lr * db[i]
        return self

    def predict(self, X):
        return self.forward(X).argmax(axis=1)

print("MLP class siap")

MLP class siap


## 4. Perbandingan split: LOSO vs Random (H5)

Reproduksi Section 5.3 paper: LOSO seharusnya menghasilkan akurasi lebih rendah (≈96,22%) daripada random frame split (≈99,68%).

In [7]:
def run_loso(X, y, subj):
    accs = []
    for held in range(1, 6):
        mask = subj == held
        Xtr, ytr = X[~mask], y[~mask]
        Xte, yte = X[mask], y[mask]
        if len(Xte) == 0 or len(np.unique(ytr)) < 2:
            continue
        model = MLP([X.shape[1], 64, 32, len(np.unique(y))])
        model.train(Xtr, ytr)
        pred = model.predict(Xte)
        accs.append((pred == yte).mean())
    return np.array(accs) if accs else np.array([])

def run_random_split(X, y, seed=0, folds=5):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X))
    n = len(X)
    accs = []
    for k in range(folds):
        te = idx[k*n//folds:(k+1)*n//folds]
        tr = np.setdiff1d(idx, te)
        model = MLP([X.shape[1], 64, 32, len(np.unique(y))])
        model.train(X[tr], y[tr])
        pred = model.predict(X[te])
        accs.append((pred == y[te]).mean())
    return np.array(accs)

print("Fungsi LOSO & random split siap")

Fungsi LOSO & random split siap


### Jalankan (butuh landmark lengkap)

```python
# 1) ekstraksi penuh (lama)
# for s in range(1,6): extract_landmarks(s)

# 2) muat semua
# X, y, subj = load_all_landmarks()
# acc_loso = run_loso(X, y, subj)
# acc_rand = run_random_split(X, y)
# print("LOSO :", acc_loso.mean().round(4), "±", acc_loso.std().round(4))
# print("Random:", acc_rand.mean().round(4), "±", acc_rand.std().round(4))
```

**Harapan (paper):** LOSO ≈ 96,22% ± 3,90; random ≈ 99,68% ± 0,08.

## 5. Ringkasan

- Dataset HANDS terverifikasi (frame, crop, label, defect path).
- Pipeline landmark MediaPipe berjalan.
- MLP + LOSO/random split siap untuk reproduksi H5.

### Catatan interpretasi (hasil sintetik)
- V0 & V2 dengan `conf=0.85` menghasilkan FA/min tinggi (>20) karena gesture idle sering salah diterima sebagai perintah.
- V1 (threshold) & V3 (wake gate) menghasilkan 0 FA karena threshold & gating menolak hampir semua non-perintah.
- Ini hanya menguji *pipeline*, bukan mengklaim performa sistem. Data final harus dari rekaman asli.